In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, str(Path("../../Tools/TACO").resolve()))

import scripts.utils, scripts.motifs

In [3]:
# =========================
# input
# =========================
offline_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_offline.tsv",
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/HCT116_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_offline.tsv"
]

complete_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/HCT116_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_complete.tsv"
]


meme_path = "/home/wangyucheng/projects/SeqEval_Bench_Data/Additional_data/human/20260123061938_JASPAR2026_combined_matrices_695344.meme"
selected_ppm_path = "/home/wangyucheng/projects/SeqEval_Bench_Data/Additional_data/human/selected_ppms.csv"
MAX_N = 50000



def infer_sep(file_path: str) -> str:
    suffix = Path(file_path).suffix.lower()
    if suffix == ".tsv":
        return "\t"
    elif suffix == ".csv":
        return "\t"
    else:
        raise ValueError(f"Unsupported file type: {file_path}")


def find_sequence_column(df: pd.DataFrame) -> str:
    candidate_cols = [
        "Sequence", "sequence", "seq", "Seq", "SEQ"
    ]
    for col in candidate_cols:
        if col in df.columns:
            return col

    for col in df.columns:
        if col.lower() == "sequence":
            return col
    for col in df.columns:
        if col.lower() == "seq":
            return col

    raise ValueError(f"Cannot find sequence column. Available columns: {list(df.columns)}")


def build_dataset_labels(file_path: str):
    path = Path(file_path)

    cell_type = path.parent.name
    dataset = path.parent.parent.name

    dataset_tag = f"{dataset}_{cell_type}"

    return dataset, cell_type, dataset_tag


def load_sequences(file_path: str, max_n: int = 50000) -> pd.DataFrame:
    sep = infer_sep(file_path)
    df = pd.read_csv(file_path, sep=sep)

    seq_col = find_sequence_column(df)
    df = df.rename(columns={seq_col: "Sequence"})

    df = df.dropna(subset=["Sequence"]).copy()

    df["Sequence"] = df["Sequence"].astype(str)

    df = df[~df["Sequence"].str.contains("N", na=False)].copy()

    if len(df) > max_n:
        df = df.sample(n=max_n, random_state=42)

    df = df.reset_index(drop=True)
    df.index = df.index.astype(str)

    return df

In [4]:
# =========================
# main
# =========================
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# Per-process state, populated once by _init_worker (fork start method)
_WORKER_STATE = {}


def _init_worker(meme_path, selected_ppm_path):
    """Initializer run once per worker process: load and filter motifs."""
    motifs, bg = scripts.motifs.read_meme(meme_path)
    sel = scripts.utils.load_csv(selected_ppm_path).Matrix_id.tolist()
    motifs = [m for m in motifs if m.name.decode() in sel]
    _WORKER_STATE["motifs"] = motifs
    _WORKER_STATE["bg"] = bg


def _process_one_file(file_path, split, max_n):
    """Process a single sequence file. Returns (file_path, success, message)."""
    try:
        level1, level2, dataset_tag = build_dataset_labels(file_path)

        # save_dir: <input file's directory> / TACO / offline or complete
        save_dir = Path(file_path).parent / "TACO" / split
        os.makedirs(save_dir, exist_ok=True)

        seqs = load_sequences(file_path, max_n=max_n)

        # motif scan
        sites = scripts.motifs.scan(seqs, _WORKER_STATE["motifs"], _WORKER_STATE["bg"])

        # motif count
        cts = scripts.motifs.calculate_motif_counts(sites)
        cts.obs = cts.obs.merge(seqs, left_index=True, right_index=True, how="left")

        if "identifier" in cts.obs.columns:
            cts.obs["identifier"] = cts.obs["identifier"].fillna("").astype(str)

        # add source metadata to obs for easier downstream tracking
        cts.obs["source_file"] = file_path
        cts.obs["level1"] = level1
        cts.obs["level2"] = level2
        cts.obs["dataset_tag"] = dataset_tag

        # save outputs
        sites_out = os.path.join(save_dir, f"{dataset_tag}__sites.csv")
        h5ad_out = os.path.join(save_dir, f"{dataset_tag}__site_counts.h5ad")

        sites.to_csv(sites_out, index=False)
        cts.write_h5ad(h5ad_out)

        return file_path, True, f"{len(seqs)} seqs -> {sites_out} ; {h5ad_out}"

    except Exception as e:
        return file_path, False, repr(e)


def run_tfbs_sites(offline_paths=None, complete_paths=None,
                   meme_path=None, selected_ppm_path=None,
                   max_n=50000, n_workers=1):
    """
    Run TFBS motif scanning on offline / complete sequence files and save
    per-site tables (sites.csv) and per-sequence motif counts (h5ad).

    Output location: TACO/offline or TACO/complete under the directory
    of each input file (created automatically if missing).

    n_workers: number of parallel worker processes; 1 means sequential.
    Each worker loads the motifs only once (via the pool initializer).
    """
    jobs = [(p, "offline") for p in (offline_paths or [])]
    jobs += [(p, "complete") for p in (complete_paths or [])]
    print(f"Total {len(jobs)} jobs, n_workers = {n_workers}")

    if n_workers <= 1:
        # Sequential execution (also useful for debugging)
        _init_worker(meme_path, selected_ppm_path)
        print(f"Loaded {len(_WORKER_STATE['motifs'])} selected motifs.")
        for i, (file_path, split) in enumerate(jobs, start=1):
            print("=" * 80)
            print(f"[{i}/{len(jobs)}] Processing ({split}): {file_path}")
            _, ok, msg = _process_one_file(file_path, split, max_n)
            print(msg if ok else f"[ERROR] {msg}")
        return

    # Parallel execution: workers inherit notebook-defined functions via fork
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(
        max_workers=n_workers,
        mp_context=ctx,
        initializer=_init_worker,
        initargs=(meme_path, selected_ppm_path),
    ) as executor:
        future_to_job = {
            executor.submit(_process_one_file, file_path, split, max_n): (file_path, split)
            for file_path, split in jobs
        }
        for done, future in enumerate(as_completed(future_to_job), start=1):
            file_path, split = future_to_job[future]
            _, ok, msg = future.result()
            status = "OK" if ok else "ERROR"
            print(f"[{done}/{len(jobs)}] [{status}] ({split}) {file_path}")
            print(f"    {msg}")




In [5]:
run_tfbs_sites(
    offline_paths=offline_path,
    complete_paths=complete_path,
    meme_path=meme_path,
    selected_ppm_path=selected_ppm_path,
    max_n=MAX_N,
    n_workers=10,
)

Total 22 jobs, n_workers = 10
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs


/tmp/ipykernel_3875742/2049580889.py:80: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=sep)


[1/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline.tsv
    27291 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/TACO/offline/DNase_GM12878__sites.csv ; /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/TACO/offline/DNase_GM12878__site_counts.h5ad
[2/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/TACO/offline/EpisomalMPRA_HepG2__sites.csv ; /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/TACO/offline/EpisomalMPRA_HepG2__site_counts.h5ad
[3/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/TACO/offline/EpisomalMPRA_A549__sites.csv ; /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/TACO/offline/EpisomalMPRA_A549__si

/tmp/ipykernel_3875742/2049580889.py:80: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=sep)


[4/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_offline.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/TACO/offline/EpisomalMPRA_K562__sites.csv ; /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/TACO/offline/EpisomalMPRA_K562__site_counts.h5ad
[5/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_offline.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/TACO/offline/EpisomalMPRA_SKNSH__sites.csv ; /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/TACO/offline/EpisomalMPRA_SKNSH__site_counts.h5ad
[6/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/HCT116_offline.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/TACO/offline/EpisomalMPRA_HCT116__sites.csv ; /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/TACO/of